In [ ]:
# Importando as Bibliotecas:
from pathlib import Path
import pandas as pd
from openpyxl.styles import Alignment, Font, PatternFill

pasta = Path('.')
saida = pasta / 'Output_Registros_Colaboradores.xlsx'

df_lista = pd.read_excel(pasta / 'data' / 'lista_colaboradores.xlsx')
df_lista['Cadastro'] = df_lista['Cadastro'].astype(str).str.strip()
df_lista['Empresa'] = df_lista['Empresa'].astype(str).str.strip()
df_lista['Nome'] = df_lista['Nome'].astype(str).str.strip()

cadastros_alvo = set(df_lista['Cadastro'])
print('Colaboradores a verificar:', len(cadastros_alvo))
display(df_lista)

In [ ]:
# Lendo o Ronda apenas com as colunas necessarias (49M+ linhas)
colunas = ['cadastro', 'nome', 'data', 'hora', 'dir', 'tipo', 'acesso', 'planta']
df_ronda = pd.read_parquet(pasta / 'data' / 'df_ronda.parquet', columns=colunas)
df_ronda['cadastro'] = df_ronda['cadastro'].astype(str).str.strip()

data_inicial = pd.Timestamp('2024-01-01')
data_final = pd.Timestamp('2026-03-31')

filtro_periodo = df_ronda['data'].between(data_inicial, data_final)
filtro_cadastros = df_ronda['cadastro'].isin(cadastros_alvo)
df_filtrado = df_ronda.loc[filtro_periodo & filtro_cadastros].copy()
df_filtrado.sort_values(['cadastro', 'data', 'hora'], inplace=True)
df_filtrado.reset_index(drop=True, inplace=True)

print('Periodo analisado:', data_inicial.strftime('%d/%m/%Y'), 'a', data_final.strftime('%d/%m/%Y'))
print('Total de registros encontrados:', len(df_filtrado))
display(df_filtrado.head())

In [ ]:
# Resumo por colaborador (entradas, saidas, dias com registro, primeiro e ultimo registro)
df_filtrado['dir'] = df_filtrado['dir'].astype(str)

agrupado = df_filtrado.groupby('cadastro')
resumo = pd.DataFrame({
    'Total Registros': agrupado.size(),
    'Entradas (E)': agrupado.apply(lambda g: (g['dir'] == 'E').sum()),
    'Saidas (S)': agrupado.apply(lambda g: (g['dir'] == 'S').sum()),
    'Dias com Registro': agrupado['data'].nunique(),
    'Primeiro Registro': agrupado['data'].min().dt.strftime('%d/%m/%Y'),
    'Ultimo Registro': agrupado['data'].max().dt.strftime('%d/%m/%Y'),
}).reset_index()

resumo = df_lista.merge(resumo, left_on='Cadastro', right_on='cadastro', how='left').drop(columns=['cadastro'])
for col in ['Total Registros', 'Entradas (E)', 'Saidas (S)', 'Dias com Registro']:
    resumo[col] = resumo[col].fillna(0).astype(int)
resumo[['Primeiro Registro', 'Ultimo Registro']] = resumo[['Primeiro Registro', 'Ultimo Registro']].fillna('-')
resumo['Possui Registro no Ronda'] = resumo['Total Registros'].gt(0).map({True: 'Sim', False: 'Nao'})

display(resumo)

In [ ]:
# Gerando o Excel final: aba de resumo + uma aba por colaborador com todos os registros
df_detalhe = df_filtrado.copy()
df_detalhe['data'] = df_detalhe['data'].dt.strftime('%d/%m/%Y')
df_detalhe['hora'] = df_detalhe['hora'].astype(str).str.slice(0, 8)
df_detalhe.rename(columns={
    'cadastro': 'Cadastro',
    'nome': 'Nome',
    'data': 'Data',
    'hora': 'Hora',
    'dir': 'Direcao',
    'tipo': 'Tipo',
    'acesso': 'Acesso',
    'planta': 'Planta',
}, inplace=True)

with pd.ExcelWriter(saida, engine='openpyxl') as writer:
    resumo.to_excel(writer, sheet_name='Resumo', index=False)
    df_detalhe.to_excel(writer, sheet_name='Registros Completos', index=False)

    for cadastro in df_lista['Cadastro']:
        nome_aba = str(cadastro)
        df_pessoa = df_detalhe[df_detalhe['Cadastro'] == cadastro]
        df_pessoa.to_excel(writer, sheet_name=nome_aba, index=False)

    wb = writer.book
    header_fill = PatternFill('solid', fgColor='1F4E78')
    header_font = Font(bold=True, color='FFFFFF')
    for ws in wb.worksheets:
        ws.freeze_panes = 'A2'
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center', vertical='center')
        for column_cells in ws.columns:
            valores = [str(cell.value) if cell.value is not None else '' for cell in column_cells]
            max_len = max(len(v) for v in valores) if valores else 10
            ws.column_dimensions[column_cells[0].column_letter].width = min(max(max_len + 2, 12), 60)

print(f'Output gerado: {saida.resolve()}')